In [ ]:
import pandas as pd
import geopandas as gpd
import sys
import importlib.util
import os


if os.getcwd().endswith("notebooks"):
    lib_path = "../lib"
else:
    lib_path = "lib"

modules = [("lib", "__init__.py")]

for module_name, module_path in modules:
    spec = importlib.util.spec_from_file_location(module_name, os.path.join(lib_path, module_path))
    module_obj = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module_obj
    spec.loader.exec_module(module_obj)

In [ ]:
if "snakemake" in locals():
    area_path = snakemake.input.area
    schedule_gpkg_path = snakemake.input.schedule_gpkg
    legs_path = snakemake.input.legs
    pt_legs_path = snakemake.input.pt_legs

    criteria = snakemake.params.criteria
    scope = snakemake.params.scope
    threshold = snakemake.params.threshold
    transit_modes = snakemake.params.transit_modes
    sampling = snakemake.params.sampling

    output_path = snakemake.output[0]
else:
    area_path = "../inputs/drn20.gpkg"
    schedule_gpkg_path = "../output/misc/network.gpkg"
    legs_path = "../output/simulations/drn_baseline/eqasim_legs.csv"
    pt_legs_path = "../output/simulations/drn_baseline/eqasim_pt.csv"

    criteria = "passenger/hour"
    scope = "off_peak"
    threshold = 50
    transit_modes = ["bus"]
    sampling = 0.25

    output_path = "../output/modified_transit_schedules/test/removed_lines.csv"

In [ ]:
assert criteria == "passenger/hour"
assert scope in ["all_day", "off_peak", "peak"]
assert isinstance(transit_modes, list)

In [ ]:
def is_within_peak(start, end):
    off_peak_periods = [(7*3600, 9*3600), (16*3600, 19*3600)]
    for p_start, p_end in off_peak_periods:
        for t in [start, end]:
            if p_start <= t <= p_end:
                return True
    return False

In [ ]:
gdf_area = gpd.read_file(area_path)
gdf_lines = gpd.read_file(schedule_gpkg_path)

df_legs = pd.read_csv(pt_legs_path, sep=";").merge(pd.read_csv(legs_path, sep=";"), how="left")

gdf_lines = gdf_lines.sjoin(gdf_area, predicate="within")

if len(transit_modes) > 0:
    df_legs = df_legs[df_legs["transit_mode"].isin(transit_modes)]
    gdf_lines = gdf_lines[gdf_lines["mode"].isin(transit_modes)]

df_legs = df_legs[df_legs["transit_line_id"].isin(gdf_lines["line_id"])]

df_legs["arrival_time"] = df_legs["departure_time"] + df_legs["travel_time"]

df_legs["departure_time"] //= 3600
df_legs["departure_time"] = df_legs["departure_time"].astype("int")
df_legs["arrival_time"] //= 3600
df_legs["arrival_time"] = df_legs["arrival_time"].astype("int")

df_legs["covered_hours"] = df_legs.apply(lambda r: list(range(r["departure_time"], r["arrival_time"]+1, 1)),axis=1)

In [ ]:
df_legs = df_legs.explode("covered_hours").rename(columns=dict(covered_hours="covered_hour"))
df_counts = df_legs[["covered_hour", "transit_line_id"]].value_counts()
df_counts.head()

In [ ]:
df_legs.head()

In [ ]:
s = df_counts.reindex(pd.MultiIndex.from_product([list(range(0, max(24, df_legs["covered_hour"].max()) + 1, 1)),
                                                  list(gdf_lines["line_id"].unique())
                                                  ],
                                                 names=["covered_hour", "transit_line_id"]))
assert s.sum() == df_counts.sum()
df_counts /= sampling
df_counts = s.reset_index()

In [ ]:
if scope == "off_peak":
    df_counts = df_counts[~df_counts["covered_hour"].apply(lambda h: is_within_peak(h, h))]
elif scope == "peak":
    df_counts = df_counts[df_counts["covered_hour"].apply(lambda h: is_within_peak(h, h))]

In [ ]:
df_counts["count"].sum()

In [ ]:
df_counts = df_counts.groupby("transit_line_id")["count"].max().reset_index()
df_selected = df_counts[df_counts["count"] < threshold]
df_selected[["transit_line_id"]].to_csv(output_path, header=True, index=False)